In [6]:
import os
import json
import pandas as pd

In [7]:
files = os.listdir("data")

print("Total Files:", len(files))
print("Sample Files:", files[:5])

Total Files: 1192
Sample Files: ['1082591.json', '1082592.json', '1082593.json', '1082594.json', '1082595.json']


In [8]:
with open(f"data/{files[0]}", "r", encoding="utf-8") as f:
    sample_match = json.load(f)

print(sample_match.keys())

dict_keys(['meta', 'info', 'innings'])


In [9]:
print(json.dumps(sample_match, indent=2)[:3000])

{
  "meta": {
    "data_version": "1.0.0",
    "created": "2017-04-06",
    "revision": 1
  },
  "info": {
    "balls_per_over": 6,
    "city": "Hyderabad",
    "dates": [
      "2017-04-05"
    ],
    "event": {
      "match_number": 1,
      "name": "Indian Premier League"
    },
    "gender": "male",
    "match_type": "T20",
    "officials": {
      "match_referees": [
        "J Srinath"
      ],
      "reserve_umpires": [
        "N Pandit"
      ],
      "tv_umpires": [
        "A Deshmukh"
      ],
      "umpires": [
        "AY Dandekar",
        "NJ Llong"
      ]
    },
    "outcome": {
      "by": {
        "runs": 35
      },
      "winner": "Sunrisers Hyderabad"
    },
    "overs": 20,
    "player_of_match": [
      "Yuvraj Singh"
    ],
    "players": {
      "Royal Challengers Bangalore": [
        "CH Gayle",
        "Mandeep Singh",
        "TM Head",
        "KM Jadhav",
        "SR Watson",
        "Sachin Baby",
        "STR Binny",
        "S Aravind",
        "TS 

In [33]:
import os
import json
import pandas as pd

all_deliveries = []

files = [f for f in os.listdir("data") if f.endswith(".json")]

print("JSON Files Found:", len(files))

for file in files:
    try:
        with open(f"data/{file}", "r", encoding="utf-8") as f:
            match = json.load(f)

        season = match["info"]["season"]

        for innings in match["innings"]:
            batting_team = innings["team"]

            for over_data in innings["overs"]:
                over_num = over_data["over"]

                for ball_num, delivery in enumerate(over_data["deliveries"], start=1):

                    dismissal_kind = None

                    if "wickets" in delivery:
                        dismissal_kind = delivery["wickets"][0]["kind"]

                    record = {
                        "match_id": file.replace(".json", ""),
                        "season": season,
                        "batting_team": batting_team,
                        "over": over_num,
                        "ball": ball_num,
                        "batter": delivery.get("batter"),
                        "bowler": delivery.get("bowler"),
                        "non_striker": delivery.get("non_striker"),
                        "batsman_runs": delivery["runs"]["batter"],
                        "extras": delivery["runs"]["extras"],
                        "total_runs": delivery["runs"]["total"],
                        "is_wicket": 1 if "wickets" in delivery else 0,
                        "dismissal_kind": dismissal_kind
                    }

                    all_deliveries.append(record)

    except Exception as e:
        print(f"Skipped file {file}: {e}")

ball_df = pd.DataFrame(all_deliveries)

ball_df.to_csv("ipl_ball_by_ball.csv", index=False)

print("Flattening Complete")
print(ball_df.head())

JSON Files Found: 1191
Flattening Complete
  match_id season         batting_team  over  ball     batter    bowler  \
0  1082591   2017  Sunrisers Hyderabad     0     1  DA Warner  TS Mills   
1  1082591   2017  Sunrisers Hyderabad     0     2  DA Warner  TS Mills   
2  1082591   2017  Sunrisers Hyderabad     0     3  DA Warner  TS Mills   
3  1082591   2017  Sunrisers Hyderabad     0     4  DA Warner  TS Mills   
4  1082591   2017  Sunrisers Hyderabad     0     5  DA Warner  TS Mills   

  non_striker  batsman_runs  extras  total_runs  is_wicket dismissal_kind  
0    S Dhawan             0       0           0          0            NaN  
1    S Dhawan             0       0           0          0            NaN  
2    S Dhawan             4       0           4          0            NaN  
3    S Dhawan             0       0           0          0            NaN  
4    S Dhawan             0       2           2          0            NaN  


In [12]:
for file in os.listdir("data"):
    print(file)

1082591.json
1082592.json
1082593.json
1082594.json
1082595.json
1082596.json
1082597.json
1082598.json
1082599.json
1082600.json
1082601.json
1082602.json
1082603.json
1082604.json
1082605.json
1082606.json
1082607.json
1082608.json
1082609.json
1082610.json
1082611.json
1082612.json
1082613.json
1082614.json
1082615.json
1082616.json
1082617.json
1082618.json
1082620.json
1082621.json
1082622.json
1082623.json
1082624.json
1082625.json
1082626.json
1082627.json
1082628.json
1082629.json
1082630.json
1082631.json
1082632.json
1082633.json
1082634.json
1082635.json
1082636.json
1082637.json
1082638.json
1082639.json
1082640.json
1082641.json
1082642.json
1082643.json
1082644.json
1082645.json
1082646.json
1082647.json
1082648.json
1082649.json
1082650.json
1136561.json
1136562.json
1136563.json
1136564.json
1136565.json
1136566.json
1136567.json
1136568.json
1136569.json
1136570.json
1136571.json
1136572.json
1136573.json
1136574.json
1136575.json
1136576.json
1136577.json
1136578.json

In [14]:
print("Total Ball Records:", len(ball_df))

Total Ball Records: 283229


In [15]:
ball_df.to_csv("ipl_ball_by_ball.csv", index=False)

In [16]:
batting_stats = ball_df.groupby("batter").agg(
    total_runs=("batsman_runs", "sum"),
    balls_played=("ball", "count"),
    innings=("match_id", "nunique")
)

batting_stats["strike_rate"] = (
    batting_stats["total_runs"] / batting_stats["balls_played"]
) * 100

batting_stats = batting_stats.sort_values(
    by="total_runs",
    ascending=False
)

print(batting_stats.head(10))

                total_runs  balls_played  innings  strike_rate
batter                                                        
V Kohli               8850          6818      263   129.803461
RG Sharma             7185          5561      270   129.203381
S Dhawan              6769          5483      221   123.454313
DA Warner             6567          4849      184   135.429986
SK Raina              5536          4177      200   132.535312
MS Dhoni              5439          4101      241   132.626189
KL Rahul              5346          4018      139   133.051269
AM Rahane             5184          4240      188   122.264151
AB de Villiers        5181          3487      170   148.580442
CH Gayle              4997          3516      141   142.121729


In [17]:
bowling_stats = ball_df.groupby("bowler").agg(
    wickets=("is_wicket", "sum"),
    balls_bowled=("ball", "count"),
    runs_given=("total_runs", "sum")
)

bowling_stats["economy"] = (
    bowling_stats["runs_given"] / bowling_stats["balls_bowled"]
) * 6

bowling_stats = bowling_stats.sort_values(
    by="wickets",
    ascending=False
)

print(bowling_stats.head(10))

            wickets  balls_bowled  runs_given   economy
bowler                                                 
YS Chahal       233          3972        5195  7.847432
B Kumar         217          4480        5691  7.621875
SP Narine       215          4511        5135  6.829971
DJ Bravo        207          3296        4436  8.075243
R Ashwin        205          4868        5721  7.051356
JJ Bumrah       204          3569        4286  7.205380
PP Chawla       201          3895        5179  7.977920
SL Malinga      188          2974        3486  7.032952
A Mishra        183          3444        4193  7.304878
RA Jadeja       182          4169        5300  7.627728


In [18]:
batting_stats.to_csv("batting_stats.csv")
bowling_stats.to_csv("bowling_stats.csv")

In [19]:
dismissals = ball_df[ball_df["is_wicket"] == 1].groupby("batter").size()

batting_stats["dismissals"] = dismissals

batting_stats["dismissals"] = batting_stats["dismissals"].fillna(0)

batting_stats["average"] = batting_stats["total_runs"] / batting_stats["dismissals"]

print(batting_stats.head(10))

                total_runs  balls_played  innings  strike_rate  dismissals  \
batter                                                                       
V Kohli               8850          6818      263   129.803461       234.0   
RG Sharma             7185          5561      270   129.203381       250.0   
S Dhawan              6769          5483      221   123.454313       194.0   
DA Warner             6567          4849      184   135.429986       164.0   
SK Raina              5536          4177      200   132.535312       168.0   
MS Dhoni              5439          4101      241   132.626189       158.0   
KL Rahul              5346          4018      139   133.051269       120.0   
AM Rahane             5184          4240      188   122.264151       171.0   
AB de Villiers        5181          3487      170   148.580442       125.0   
CH Gayle              4997          3516      141   142.121729       128.0   

                  average  
batter                     
V Kohli

In [20]:
boundary_runs = ball_df[
    ball_df["batsman_runs"].isin([4, 6])
].groupby("batter")["batsman_runs"].sum()

batting_stats["boundary_runs"] = boundary_runs

batting_stats["boundary_runs"] = batting_stats["boundary_runs"].fillna(0)

batting_stats["boundary_percent"] = (
    batting_stats["boundary_runs"] / batting_stats["total_runs"]
) * 100

print(batting_stats.head(10))

                total_runs  balls_played  innings  strike_rate  dismissals  \
batter                                                                       
V Kohli               8850          6818      263   129.803461       234.0   
RG Sharma             7185          5561      270   129.203381       250.0   
S Dhawan              6769          5483      221   123.454313       194.0   
DA Warner             6567          4849      184   135.429986       164.0   
SK Raina              5536          4177      200   132.535312       168.0   
MS Dhoni              5439          4101      241   132.626189       158.0   
KL Rahul              5346          4018      139   133.051269       120.0   
AM Rahane             5184          4240      188   122.264151       171.0   
AB de Villiers        5181          3487      170   148.580442       125.0   
CH Gayle              4997          3516      141   142.121729       128.0   

                  average  boundary_runs  boundary_percent  
ba

In [21]:
dot_balls = ball_df[
    ball_df["batsman_runs"] == 0
].groupby("batter").size()

batting_stats["dot_balls"] = dot_balls

batting_stats["dot_balls"] = batting_stats["dot_balls"].fillna(0)

batting_stats["dot_ball_percent"] = (
    batting_stats["dot_balls"] / batting_stats["balls_played"]
) * 100

print(batting_stats.head(10))

                total_runs  balls_played  innings  strike_rate  dismissals  \
batter                                                                       
V Kohli               8850          6818      263   129.803461       234.0   
RG Sharma             7185          5561      270   129.203381       250.0   
S Dhawan              6769          5483      221   123.454313       194.0   
DA Warner             6567          4849      184   135.429986       164.0   
SK Raina              5536          4177      200   132.535312       168.0   
MS Dhoni              5439          4101      241   132.626189       158.0   
KL Rahul              5346          4018      139   133.051269       120.0   
AM Rahane             5184          4240      188   122.264151       171.0   
AB de Villiers        5181          3487      170   148.580442       125.0   
CH Gayle              4997          3516      141   142.121729       128.0   

                  average  boundary_runs  boundary_percent  dot

In [22]:
player_match_runs = ball_df.groupby(
    ["batter", "match_id"]
)["batsman_runs"].sum().reset_index()

consistency = player_match_runs.groupby("batter")["batsman_runs"].std()

batting_stats["consistency_score"] = consistency

print(batting_stats.head(10))

                total_runs  balls_played  innings  strike_rate  dismissals  \
batter                                                                       
V Kohli               8850          6818      263   129.803461       234.0   
RG Sharma             7185          5561      270   129.203381       250.0   
S Dhawan              6769          5483      221   123.454313       194.0   
DA Warner             6567          4849      184   135.429986       164.0   
SK Raina              5536          4177      200   132.535312       168.0   
MS Dhoni              5439          4101      241   132.626189       158.0   
KL Rahul              5346          4018      139   133.051269       120.0   
AM Rahane             5184          4240      188   122.264151       171.0   
AB de Villiers        5181          3487      170   148.580442       125.0   
CH Gayle              4997          3516      141   142.121729       128.0   

                  average  boundary_runs  boundary_percent  dot

In [23]:
death_df = ball_df[ball_df["over"] >= 15]

death_stats = death_df.groupby("batter").agg(
    death_runs=("batsman_runs", "sum"),
    death_balls=("ball", "count")
)

death_stats["death_sr"] = (
    death_stats["death_runs"] / death_stats["death_balls"]
) * 100

print(death_stats.sort_values("death_runs", ascending=False).head(10))

                death_runs  death_balls    death_sr
batter                                             
MS Dhoni              3468         2083  166.490639
KA Pollard            2032         1270  160.000000
KD Karthik            1904         1151  165.421373
AB de Villiers        1868          867  215.455594
RA Jadeja             1831         1282  142.823713
RG Sharma             1527          876  174.315068
V Kohli               1525          852  178.990610
HH Pandya             1448          896  161.607143
AD Russell            1432          795  180.125786
DA Miller             1429          864  165.393519


In [24]:
powerplay_df = ball_df[ball_df["over"] <= 5]

powerplay_stats = powerplay_df.groupby("batter").agg(
    pp_runs=("batsman_runs", "sum"),
    pp_balls=("ball", "count")
)

powerplay_stats["pp_sr"] = (
    powerplay_stats["pp_runs"] / powerplay_stats["pp_balls"]
) * 100

print(powerplay_stats.sort_values("pp_runs", ascending=False).head(10))

              pp_runs  pp_balls       pp_sr
batter                                     
S Dhawan         3415      2862  119.322152
DA Warner        3318      2501  132.666933
V Kohli          3229      2616  123.432722
AM Rahane        2637      2199  119.918145
RG Sharma        2481      2032  122.096457
CH Gayle         2405      1851  129.929768
F du Plessis     2281      1713  133.158202
G Gambhir        2277      1945  117.069409
KL Rahul         2241      1807  124.017709
RV Uthappa       2037      1697  120.035357


In [25]:
batting_stats.to_csv("advanced_batting_stats.csv")
death_stats.to_csv("death_over_stats.csv")
powerplay_stats.to_csv("powerplay_stats.csv")

In [26]:
boundary_stats = ball_df[
    ball_df["batsman_runs"].isin([4, 6])
].groupby("batter").agg(
    boundaries=("batsman_runs", "count"),
    boundary_runs=("batsman_runs", "sum")
)

boundary_stats = boundary_stats.sort_values(
    by="boundaries",
    ascending=False
)

print(boundary_stats.head(10))

                boundaries  boundary_runs
batter                                   
V Kohli               1092           4966
RG Sharma              964           4478
S Dhawan               921           3990
DA Warner              899           4068
CH Gayle               767           3786
SK Raina               710           3248
KL Rahul               679           3142
AB de Villiers         667           3174
RV Uthappa             663           3016
AM Rahane              655           2884


In [27]:
player_match_runs = ball_df.groupby(
    ["batter", "match_id"]
)["batsman_runs"].sum().reset_index()

consistency_stats = player_match_runs.groupby("batter").agg(
    avg_runs=("batsman_runs", "mean"),
    std_dev=("batsman_runs", "std")
)

consistency_stats["consistency_index"] = (
    consistency_stats["avg_runs"] / consistency_stats["std_dev"]
)

consistency_stats = consistency_stats.sort_values(
    by="consistency_index",
    ascending=False
)

print(consistency_stats.head(10))

                avg_runs   std_dev  consistency_index
batter                                               
XC Bartlett         11.0  0.000000                inf
JP Behrendorff       3.0  0.000000                inf
SB Joshi             3.0  0.000000                inf
R Minz               3.0  0.000000                inf
Akash Madhwal        4.0  0.000000                inf
CRD Fernando         2.0  0.000000                inf
MA Khote             8.0  1.000000           8.000000
VH Zol              14.5  2.121320           6.835366
T Banton             9.0  1.414214           6.363961
Shivang Kumar        4.5  0.707107           6.363961


In [28]:
death_df = ball_df[ball_df["over"] >= 15]

death_specialists = death_df.groupby("batter").agg(
    death_runs=("batsman_runs", "sum"),
    death_balls=("ball", "count")
)

death_specialists["death_sr"] = (
    death_specialists["death_runs"] /
    death_specialists["death_balls"]
) * 100

death_specialists = death_specialists.sort_values(
    by="death_runs",
    ascending=False
)

print(death_specialists.head(10))

                death_runs  death_balls    death_sr
batter                                             
MS Dhoni              3468         2083  166.490639
KA Pollard            2032         1270  160.000000
KD Karthik            1904         1151  165.421373
AB de Villiers        1868          867  215.455594
RA Jadeja             1831         1282  142.823713
RG Sharma             1527          876  174.315068
V Kohli               1525          852  178.990610
HH Pandya             1448          896  161.607143
AD Russell            1432          795  180.125786
DA Miller             1429          864  165.393519


In [29]:
powerplay_df = ball_df[ball_df["over"] <= 5]

powerplay_performers = powerplay_df.groupby("batter").agg(
    pp_runs=("batsman_runs", "sum"),
    pp_balls=("ball", "count")
)

powerplay_performers["pp_sr"] = (
    powerplay_performers["pp_runs"] /
    powerplay_performers["pp_balls"]
) * 100

powerplay_performers = powerplay_performers.sort_values(
    by="pp_runs",
    ascending=False
)

print(powerplay_performers.head(10))

              pp_runs  pp_balls       pp_sr
batter                                     
S Dhawan         3415      2862  119.322152
DA Warner        3318      2501  132.666933
V Kohli          3229      2616  123.432722
AM Rahane        2637      2199  119.918145
RG Sharma        2481      2032  122.096457
CH Gayle         2405      1851  129.929768
F du Plessis     2281      1713  133.158202
G Gambhir        2277      1945  117.069409
KL Rahul         2241      1807  124.017709
RV Uthappa       2037      1697  120.035357


In [30]:
matchup_stats = ball_df.groupby(
    ["batter", "bowler"]
).agg(
    runs=("batsman_runs", "sum"),
    balls=("ball", "count"),
    dismissals=("is_wicket", "sum")
)

matchup_stats["strike_rate"] = (
    matchup_stats["runs"] / matchup_stats["balls"]
) * 100

print(matchup_stats.head(20))

                                runs  balls  dismissals  strike_rate
batter         bowler                                               
A Ashish Reddy A Nehra             7      9           1    77.777778
               AB Dinda            9      7           0   128.571429
               AD Mathews         25     12           0   208.333333
               AD Russell          4      3           1   133.333333
               Anureet Singh       2      2           0   100.000000
               Azhar Mahmood       2      3           0    66.666667
               B Kumar             1      1           0   100.000000
               BW Hilfenhaus       2      2           1   100.000000
               CH Morris          11      5           0   220.000000
               DJ Bravo           24     13           1   184.615385
               Harbhajan Singh    11      8           0   137.500000
               Imran Tahir         2      2           0   100.000000
               J Botha            

In [31]:
print(matchup_stats.loc[("V Kohli", "JJ Bumrah")])

runs           164.000000
balls          111.000000
dismissals       5.000000
strike_rate    147.747748
Name: (V Kohli, JJ Bumrah), dtype: float64


In [32]:
pip install streamlit plotly

     ---------------------------------------- 0.0/44.3 kB ? eta -:--:--
     ------------------------------------ --- 41.0/44.3 kB 1.9 MB/s eta 0:00:01
     -------------------------------------- 44.3/44.3 kB 724.5 kB/s eta 0:00:00
  Using cached jinja2-3.1.6-py3-none-any.whl.metadata (2.9 kB)
  Using cached jsonschema-4.26.0-py3-none-any.whl.metadata (7.6 kB)
     ---------------------------------------- 0.0/41.7 kB ? eta -:--:--
     ---------------------------------------- 41.7/41.7 kB 1.0 MB/s eta 0:00:00
  Using cached idna-3.11-py3-none-any.whl.metadata (8.4 kB)
  Using cached urllib3-2.6.3-py3-none-any.whl.metadata (6.9 kB)
  Using cached certifi-2026.2.25-py3-none-any.whl.metadata (2.5 kB)
  Using cached jsonschema_specifications-2025.9.1-py3-none-any.whl.metadata (2.9 kB)
  Using cached referencing-0.37.0-py3-none-any.whl.metadata (2.8 kB)
   ---------------------------------------- 0.0/9.1 MB ? eta -:--:--
    --------------------------------------- 0.2/9.1 MB 3.3 MB/s eta 0:


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [1]:
import pandas as pd

ball_df = pd.read_csv("ipl_ball_by_ball.csv")

# ---------------- BATTING PER MATCH ----------------
player_match_runs = ball_df.groupby(
    ["batter", "match_id"]
)["batsman_runs"].sum().reset_index()

# 50s and 100s
fifties = player_match_runs[
    (player_match_runs["batsman_runs"] >= 50) &
    (player_match_runs["batsman_runs"] < 100)
].groupby("batter").size()

hundreds = player_match_runs[
    player_match_runs["batsman_runs"] >= 100
].groupby("batter").size()

# Best Batting
best_batting = player_match_runs.groupby("batter")["batsman_runs"].max()

# ---------------- BOWLING ----------------
wickets = ball_df.groupby("bowler")["is_wicket"].sum()

bowler_match_wickets = ball_df.groupby(
    ["bowler", "match_id"]
)["is_wicket"].sum().reset_index()

best_bowling = bowler_match_wickets.groupby("bowler")["is_wicket"].max()

# ---------------- MERGE INTO BATTING TABLE ----------------
batting = pd.read_csv("advanced_batting_stats.csv")

batting["total_50s"] = batting["batter"].map(fifties).fillna(0)
batting["total_100s"] = batting["batter"].map(hundreds).fillna(0)
batting["best_batting"] = batting["batter"].map(best_batting).fillna(0)
batting["total_wickets"] = batting["batter"].map(wickets).fillna(0)
batting["best_bowling"] = batting["batter"].map(best_bowling).fillna(0)

batting.to_csv("advanced_batting_stats.csv", index=False)

print("Updated stats CSV created successfully.")

C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_18504\897916644.py:3: DtypeWarning: Columns (0: season) have mixed types. Specify dtype option on import or set low_memory=False.
  ball_df = pd.read_csv("ipl_ball_by_ball.csv")


Updated stats CSV created successfully.


In [2]:
import pandas as pd

ball_df = pd.read_csv("ipl_ball_by_ball.csv")
batting = pd.read_csv("advanced_batting_stats.csv")

# =========================================
# BEST BATTING RUNS + BALLS
# =========================================
player_match_runs = ball_df.groupby(
    ["batter", "match_id"]
)["batsman_runs"].sum().reset_index()

player_match_balls = ball_df.groupby(
    ["batter", "match_id"]
).size().reset_index(name="balls")

best_batting_records = []

for batter in player_match_runs["batter"].unique():

    batter_matches = player_match_runs[
        player_match_runs["batter"] == batter
    ]

    best_score_row = batter_matches.loc[
        batter_matches["batsman_runs"].idxmax()
    ]

    best_match = best_score_row["match_id"]

    balls = player_match_balls[
        (player_match_balls["batter"] == batter) &
        (player_match_balls["match_id"] == best_match)
    ]["balls"].values[0]

    best_batting_records.append({
        "batter": batter,
        "best_batting_runs": best_score_row["batsman_runs"],
        "best_batting_balls": balls
    })

best_batting_df = pd.DataFrame(best_batting_records)

# =========================================
# BEST BOWLING WICKETS + RUNS
# =========================================
bowler_match_stats = ball_df.groupby(
    ["bowler", "match_id"]
).agg(
    wickets=("is_wicket", "sum"),
    runs_given=("total_runs", "sum")
).reset_index()

best_bowling_records = []

for bowler in bowler_match_stats["bowler"].unique():

    bowler_matches = bowler_match_stats[
        bowler_match_stats["bowler"] == bowler
    ]

    best_match = bowler_matches.sort_values(
        by=["wickets", "runs_given"],
        ascending=[False, True]
    ).iloc[0]

    best_bowling_records.append({
        "batter": bowler,
        "best_bowling_wickets": best_match["wickets"],
        "best_bowling_runs": best_match["runs_given"]
    })

best_bowling_df = pd.DataFrame(best_bowling_records)

# =========================================
# MERGE NEW COLUMNS
# =========================================
batting = batting.merge(
    best_batting_df,
    on="batter",
    how="left"
)

batting = batting.merge(
    best_bowling_df,
    on="batter",
    how="left"
)

# SAVE UPDATED CSV
batting.fillna(0, inplace=True)

batting.to_csv("advanced_batting_stats.csv", index=False)

print("CSV UPDATED SUCCESSFULLY")

C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_18504\1212976571.py:3: DtypeWarning: Columns (0: season) have mixed types. Specify dtype option on import or set low_memory=False.
  ball_df = pd.read_csv("ipl_ball_by_ball.csv")


CSV UPDATED SUCCESSFULLY


In [1]:
pip install mysql-connector-python


   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.5 MB ? eta -:--:--
   ---------------------------------------- 0.0/16.5 MB 325.1 kB/s eta 0:00:51
   ---------------------------------------- 0.1/16.5 MB 653.6 kB/s eta 0:00:26
    --------------------------------------- 0.3/16.5 MB 1.4 MB/s eta 0:00:12
   - -------------------------------------- 0.6/16.5 MB 2.6 MB/s eta 0:00:07
   -- ------------------------------------- 1.0/16.5 MB 3.9 MB/s eta 0:00:04
   -- ------------------------------------- 1.0/16.5 MB 3.9 MB/s eta 0:00:04
   -- ------------------------------------- 1.0/16.5 MB 3.9 MB/s eta 0:00:04
   -- ------------------------------------- 1.0/16.5 MB 3.9 MB/s eta 0:00:04
   -- ------------------------------------- 1.2/16.5 MB 2.6 MB/s eta 0:00:06
   ---- ----------------------------------- 1.8/16.5 MB 3.6 MB/s eta 0:00:05
   ----- -------


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
print(batting.columns)
print(len(batting.columns))

Index(['batter', 'total_runs', 'balls_played', 'innings', 'strike_rate',
       'dismissals', 'average', 'boundary_runs', 'boundary_percent',
       'dot_balls', 'dot_ball_percent', 'consistency_score', 'total_50s',
       'total_100s', 'best_batting', 'total_wickets', 'best_bowling',
       'best_batting_runs', 'best_batting_balls', 'best_bowling_wickets',
       'best_bowling_runs'],
      dtype='str')
21


In [ ]:
import pandas as pd
import numpy as np
import mysql.connector


batting = pd.read_csv("advanced_batting_stats.csv")
ball_df = pd.read_csv("ipl_ball_by_ball.csv")


ball_df = ball_df.rename(columns={"over": "over_num"})

# ================= CLEAN DATA =================


batting.replace([np.inf, -np.inf], 0, inplace=True)
batting.fillna(0, inplace=True)


num_cols = ball_df.select_dtypes(include=['number']).columns
ball_df[num_cols] = ball_df[num_cols].fillna(0)

str_cols = ball_df.select_dtypes(include=['object', 'string']).columns
ball_df[str_cols] = ball_df[str_cols].fillna("")


ball_df["season"] = pd.to_numeric(ball_df["season"], errors="coerce")
ball_df["season"] = ball_df["season"].fillna(0).astype(int)

batting = batting[[
    'batter','total_runs','balls_played','innings','strike_rate','average',
    'total_50s','total_100s','boundary_runs','boundary_percent',
    'dot_ball_percent','consistency_score','dismissals','total_wickets',
    'best_batting_runs','best_batting_balls',
    'best_bowling_wickets','best_bowling_runs'
]]

ball_df = ball_df[[
    'match_id','season','batting_team','over_num','ball',
    'batter','bowler','batsman_runs','extras','total_runs',
    'is_wicket','dismissal_kind'
]]

# ================= CONNECT TO MYSQL =================
conn = mysql.connector.connect(
    host="localhost",
    user="root",
    password="Harsh@425",   
    database="ipl"
)

conn.autocommit = True
cursor = conn.cursor()


cursor.execute("TRUNCATE TABLE batting_stats")
cursor.execute("TRUNCATE TABLE ball_by_ball")


cursor.executemany("""
    INSERT INTO batting_stats (
        batter,total_runs,balls_played,innings,strike_rate,average,
        total_50s,total_100s,boundary_runs,boundary_percent,
        dot_ball_percent,consistency_score,dismissals,total_wickets,
        best_batting_runs,best_batting_balls,
        best_bowling_wickets,best_bowling_runs
    )
    VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
""", batting.values.tolist())

print("Batting data inserted")


batch_size = 5000
data = ball_df.values.tolist()

for i in range(0, len(data), batch_size):
    batch = data[i:i+batch_size]
    
    cursor.executemany("""
        INSERT INTO ball_by_ball (
            match_id,season,batting_team,over_num,ball,
            batter,bowler,batsman_runs,extras,total_runs,
            is_wicket,dismissal_kind
        )
        VALUES (%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s,%s)
    """, batch)
    
    print(f"Inserted {i + len(batch)} rows")


conn.close()

print("All data successfully inserted into MySQL")

C:\Users\KIIT0001\AppData\Local\Temp\ipykernel_5860\133143269.py:7: DtypeWarning: Columns (0: season) have mixed types. Specify dtype option on import or set low_memory=False.
  ball_df = pd.read_csv("ipl_ball_by_ball.csv")


Batting data inserted
Inserted 5000 rows
Inserted 10000 rows
Inserted 15000 rows
Inserted 20000 rows
Inserted 25000 rows
Inserted 30000 rows
Inserted 35000 rows
Inserted 40000 rows
Inserted 45000 rows
Inserted 50000 rows
Inserted 55000 rows
Inserted 60000 rows
Inserted 65000 rows
Inserted 70000 rows
Inserted 75000 rows
Inserted 80000 rows
Inserted 85000 rows
Inserted 90000 rows
Inserted 95000 rows
Inserted 100000 rows
Inserted 105000 rows
Inserted 110000 rows
Inserted 115000 rows
Inserted 120000 rows
Inserted 125000 rows
Inserted 130000 rows
Inserted 135000 rows
Inserted 140000 rows
Inserted 145000 rows
Inserted 150000 rows
Inserted 155000 rows
Inserted 160000 rows
Inserted 165000 rows
Inserted 170000 rows
Inserted 175000 rows
Inserted 180000 rows
Inserted 185000 rows
Inserted 190000 rows
Inserted 195000 rows
Inserted 200000 rows
Inserted 205000 rows
Inserted 210000 rows
Inserted 215000 rows
Inserted 220000 rows
Inserted 225000 rows
Inserted 230000 rows
Inserted 235000 rows
Inserted 24